In [6]:
# Install packages
# %pip install requests
# %pip install bs4
# %pip install googlesearch-python


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... done
  Created wheel for googlesearch-python: filename=googlesearch_python-1.2.3-py3-none-any.whl size=4229 sha256=aabf0357a49a6a798fd985e3327f817a844deef28704a6f70017547b4043a8ab
  Stored in directory: /Users/modekurt/Library/Caches/pip/wheels/be/78/a3/d80e85ac9551489cd9c44f59a0bc5972e79e680ce9cc6055ca
Successfully built googlesearch-python
Note: you may need to restart the kernel to use updated packages.


In [27]:
import requests
from bs4 import BeautifulSoup
from googlesearch import search
import os
import pandas as pd
import numpy

In [5]:
# Extract Municipal Website from City Wikipedia Page
# Example: Sacramento, CA

def get_sacramento_municipal_website_link():
    # Wikipedia URL for Sacramento
    wikipedia_url = 'https://en.wikipedia.org/wiki/Sacramento,_California'

    # Send a request to the Wikipedia page
    response = requests.get(wikipedia_url)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content of the page
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the right panel section
        right_panel = soup.find('table', {'class': 'infobox'})

        # Check if the right panel section exists
        if right_panel:
            # Find the row containing the 'Website' label
            website_row = right_panel.find('th', string='Website')

            # Check if the row is found
            if website_row:
                # Get the link from the corresponding 'td' element
                link = website_row.find_next('td').find('a', href=True)

                # Check if a link is found
                if link:
                    return link['href']
                else:
                    return "Municipal website link not found in the right panel."
            else:
                return "'Website' label not found in the right panel."
        else:
            return "Right panel section not found on the Wikipedia page."
    else:
        return f"Error accessing the Wikipedia page. Status code: {response.status_code}"

# Example usage for Sacramento
sacramento_municipal_website_link = get_sacramento_municipal_website_link()

print(f"Municipal Website Link for Sacramento: {sacramento_municipal_website_link}")


Municipal Website Link for Sacramento: http://cityofsacramento.org


In [15]:
#Now search on google for climate action plans using extracted link

def download_search_results(query, city_name, city_code, num_results):
    # Perform Google search and get the first 'num_results' URLs
    #I think th
    search_results = list(search(query, num_results=num_results))
    print(search_results)

    # Create a directory to save the PDFs
    folder_path = 'plans/'+city_code+'-'+city_name
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    # Download content from each URL
    for i, url in enumerate(search_results):
        try:
            response = requests.get(url)
            # print(requests.head(url).headers['last-modified'])

            # Save the content to a file (you can modify this part based on your needs)
            if url.endswith('.pdf'):
                pdf_name = os.path.join(folder_path, city_code+'-[date]-'+os.path.basename(url))

                with open(pdf_name, 'wb') as pdf_file:
                    pdf_file.write(response.content)
                    print(f"Downloaded result {i + 1}: {url}")
        except Exception as e:
            print(f"Error downloading result {i + 1}: {url}\nError: {e}")

# Example usage
# search_query = 'Mendoza' + 'climate adaptation plan' + 'filetype:pdf'
# download_search_results(search_query, 'sacramento', '12345', num_results=1)

In [28]:
#TODO: get and check translations 
#TODO: look to literature for other search queries
search_queries_A = {'english': ['climate adaptation plan'], 'spanish': ['Plan de Adaptación al Cambio Climatico'], 
                    'german': ['Anpassung an den Klimawandel'], 'french': ['plan d\'adaptation climatique'], 'portuguese': ['Plano de Adaptação às Alterações Climáticas'] } 
city_list = pd.read_csv('output/capital_and_1mill_pop_cities_lang.csv')

def plan_search(row):
    print(row)
    # TODO: implement language switch query = search_queries[row['language'].tolower()] + row['City']
    city_name = row['city']
    city_lang = 'english' if numpy.isnan(row['language']) else row['language']
    print(city_lang)
    query = city_name + search_queries_A[city_lang.lower()][0]
    city_code = str(row['city_code'])
    download_search_results(query, city_name, city_code, 10)
    return 

#look for searches for each city
searched_plans = city_list.apply(plan_search, axis=1)

Unnamed: 0                0
country         Afghanistan
country_code              4
city                  Kabul
city_code             20002
Latitude          34.528887
Longitude          69.17246
population          4011.77
language                NaN
Name: 0, dtype: object
english
['https://www4.unfccc.int/sites/NAPC/Country%20Documents/Parties/napa-afghanistan-final.pdf', 'https://unfccc.int/sites/default/files/resource/Scanned-color.pdf', 'https://www.adaptation-undp.org/explore/asia-and-pacific/afghanistan', 'https://wedocs.unep.org/bitstream/handle/20.500.11822/22447/Report_CC_Governance_Afghanistan_EN_v2.pdf?sequence=1&isAllowed=y', 'https://tech-action.unepccc.org/wp-content/uploads/sites/2/2020/08/tna-adaptation-report-afghanistan.pdf', 'https://www.nepa.gov.af/service3', 'https://www.adpc.net/icare/wp-content/uploads/2022/01/1-1.pdf', 'https://www.sei.org/projects-and-tools/projects/afghanistan-climate-change-adaptation-action-plans/', 'https://www.undp.org/sites/g/files/zskgk

KeyboardInterrupt: 